In [42]:
#Withdrawn IPOs
import pandas as pd
import numpy as np
import io
import requests

In [43]:
url = "https://www.iposcoop.com/ipos-recently-filed/"

response = requests.get(url)

tables = pd.read_html(io.StringIO(response.text))

len(tables)

1

In [44]:
tables[0].head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
1,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Friday,S/O
2,2026-09-24,Hacker Interstellar,SOUD,Kingswood Capital Markets,3.90,$10.50,$11.50,$42.90,TBA,S/O
3,2026-09-24,New Iceland Arctic Acquisition Corp.,NIAAU,Chardan,12.50,$10.00,$10.00,$125.00,Friday,S/O
4,2026-09-23,Frazier Life Sciences Acquisition II (Stock-Only),FLSC,Jefferies,7.50,$10.00,$10.00,$75.00,TBA,S/O


In [45]:
df = tables[0].copy()

df.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
1,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Friday,S/O
2,2026-09-24,Hacker Interstellar,SOUD,Kingswood Capital Markets,3.90,$10.50,$11.50,$42.90,TBA,S/O
3,2026-09-24,New Iceland Arctic Acquisition Corp.,NIAAU,Chardan,12.50,$10.00,$10.00,$125.00,Friday,S/O
4,2026-09-23,Frazier Life Sciences Acquisition II (Stock-Only),FLSC,Jefferies,7.50,$10.00,$10.00,$75.00,TBA,S/O


In [46]:
withdrawn = df[
    df["Expected To Trade"].astype(str).str.strip().eq("Withdrawn")
].copy()

withdrawn.shape

(34, 10)

In [47]:
def classify_company(name):

    name = str(name)

    if "Technologies" in name:
        return "Technologies"

    elif (
        "Acquisition Corp" in name
        or "Acquisition Corporation" in name
        or "Corp" in name
    ):
        return "Acquisition Corp"

    elif (
        "Inc" in name
        or "Incorporated" in name
    ):
        return "Inc."

    elif "Group" in name:
        return "Group"

    elif (
        "Ltd" in name
        or "Limited" in name
    ):
        return "Limited"

    elif (
        "Holdings" in name
        or "Holding" in name
    ):
        return "Holdings"

    else:
        return "Other"

In [48]:
withdrawn["Company Type"] = withdrawn["Company"].apply(
    classify_company
)

In [49]:
def extract_price(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value == "-":
        return np.nan

    value = value.replace("$", "").replace(",", "")

    try:
        return float(value)
    except:
        return np.nan

In [50]:
withdrawn["Price Low Num"] = withdrawn["Price Low"].apply(
    extract_price
)

withdrawn["Price High Num"] = withdrawn["Price High"].apply(
    extract_price
)

In [51]:
withdrawn["Avg_price"] = withdrawn[
    ["Price Low Num", "Price High Num"]
].mean(axis=1)

In [52]:
withdrawn["Shares (millions)"] = pd.to_numeric(
    withdrawn["Shares (millions)"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False),
    errors="coerce"
)

In [53]:
withdrawn["Est $ Vol (millions)"] = pd.to_numeric(
    withdrawn["Est $ Vol (millions)"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False),
    errors="coerce"
)

In [54]:
calculated_value = (
    withdrawn["Shares (millions)"]
    * withdrawn["Avg_price"]
)

In [55]:
withdrawn["Shares_offered_value"] = calculated_value.fillna(
    withdrawn["Est $ Vol (millions)"]
)

In [56]:
withdrawal_summary = (
    withdrawn
    .groupby("Company Type")["Shares_offered_value"]
    .sum()
    .sort_values(ascending=False)
)

withdrawal_summary

Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             219.2500
Technologies        184.9000
Group                56.5750
Name: Shares_offered_value, dtype: float64

In [57]:
withdrawal_summary.idxmax()

'Acquisition Corp'

In [58]:
withdrawal_summary.max()

499.985

In [59]:
#IPO Sharpe Ratio
#├── Q3 — Fixed Months Holding Strategy
#├── Q4 — RSI Trading Strategy
#└── Q5 — Improving IPO Strategy
import pandas as pd
import numpy as np
import yfinance as yf
import io

# --------------------------------------------------
# 1. Load 2025 IPO data
# --------------------------------------------------

url = "https://www.iposcoop.com/2025-pricings/"

tables = pd.read_html(url)

# Check available tables
for i, table in enumerate(tables):
    print(i, table.shape)

0 (231, 10)


In [60]:
ipos = tables[0].copy()

print(ipos.head())
print(ipos.columns)

                                 Company Symbol           Industry  \
0  Vine Hill Capital Investment Corp. II  VHCPU        Blank Check   
1                         Andersen Group   ANDG  Consumer Services   
2                           Medline Inc.   MDLN        Health Care   
3                      Wealthfront Corp.   WLTH         Financials   
4                Lumexa Imaging Holdings   LMRI        Health Care   

   Offer Date  Shares (millions) Offer Price 1st Day Close Current Price  \
0  12/18/2025               20.0      $10.00         $0.00         $0.00   
1  12/17/2025               11.0      $16.00        $23.50        $23.79   
2  12/17/2025              216.0      $29.00        $41.00        $43.83   
3  12/12/2025               34.6      $14.00        $14.19         $8.49   
4  12/11/2025               25.0      $18.50        $18.52        $14.80   

    Return SCOOP Rating  
0    0.00%          S/O  
1   48.69%          S/O  
2   51.14%          S/O  
3  -39.36%        

In [61]:
ipos["Offer Date"] = pd.to_datetime(
    ipos["Offer Date"],
    errors="coerce"
)

In [62]:
print(ipos.columns)

Index(['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)',
       'Offer Price', '1st Day Close', 'Current Price', 'Return',
       'SCOOP Rating'],
      dtype='object')


In [63]:
ipos["Return_num"] = (
    ipos["Return"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

ipos = ipos[
    (ipos["Offer Date"] < "2025-09-01") &
    (ipos["Return_num"] != 0)
].copy()

print("Number of IPOs:", len(ipos))

Number of IPOs: 146


In [64]:
print("Number of IPOs:", len(ipos))

Number of IPOs: 146


In [65]:
print(ipos.columns)

Index(['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)',
       'Offer Price', '1st Day Close', 'Current Price', 'Return',
       'SCOOP Rating', 'Return_num'],
      dtype='object')


In [66]:
tickers = ipos["Symbol"].dropna().unique().tolist()

print("Number of tickers:", len(tickers))

Number of tickers: 146


In [67]:
stocks_df = yf.download(
    tickers,
    start="2025-01-01",
    end="2026-09-12",
    group_by="column",
    auto_adjust=False,
    threads=True
)

[*********             18%                       ]  26 of 146 completed$MJID: possibly delisted; no timezone found
[************          25%                       ]  37 of 146 completed$EMPG: possibly delisted; no timezone found
[****************      34%                       ]  49 of 146 completed$WGRX: possibly delisted; no timezone found
[****************      34%                       ]  50 of 146 completed$PTNM: possibly delisted; no timezone found
[*******************   39%                       ]  57 of 146 completed$AGH: possibly delisted; no timezone found
[*******************   40%                       ]  58 of 146 completed$CEPT: possibly delisted; no timezone found
[********************* 43%                       ]  63 of 146 completed$CAEP: possibly delisted; no timezone found
[**********************45%                       ]  66 of 146 completed$AHL: possibly delisted; no timezone found
[**********************46%                       ]  67 of 146 completed$SKBL: poss

In [68]:
if isinstance(stocks_df.columns, pd.MultiIndex):
    stocks_df.columns = stocks_df.columns.get_level_values(0)

In [69]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download all tickers at once
stocks_raw = yf.download(
    tickers,
    start="2025-01-01",
    end="2026-09-12",
    auto_adjust=False,
    progress=False,
    threads=True
)

# Get Close prices
close_prices = stocks_raw["Close"].copy()

print("Downloaded data shape:", close_prices.shape)

$MJID: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$SKBL: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found

14 Failed downloads:
['MJID', 'EMPG', 'PTNM', 'WGRX', 'CEPT', 'AGH', 'CAEP', 'SKBL', 'MTSR', 'AHL', 'SDM', 'MCTR', 'EPWK', 'TBH']: possibly delisted; no timezone found


Downloaded data shape: (424, 146)


In [71]:
stocks_df = (
    close_prices
    .reset_index()
    .melt(
        id_vars="Date",
        var_name="Ticker",
        value_name="Close"
    )
)

stocks_df = stocks_df.dropna(subset=["Close"])

stocks_df = stocks_df.sort_values(["Ticker", "Date"])

print(stocks_df.shape)
print(stocks_df.head())

(44821, 3)
         Date Ticker      Close
14 2025-01-24   AAPG  17.379999
15 2025-01-27   AAPG  17.250000
16 2025-01-28   AAPG  17.200001
17 2025-01-29   AAPG  17.330000
18 2025-01-30   AAPG  17.600000


In [75]:
stocks_df["growth_252d"] = (
    stocks_df.groupby("Ticker")["Close"]
    .transform(lambda x: x / x.shift(252))
)

In [76]:
stocks_df["volatility"] = (
    stocks_df.groupby("Ticker")["Close"]
    .transform(
        lambda x: x.rolling(30).std() * np.sqrt(252)
    )
)

In [77]:
stocks_df["Sharpe"] = (
    (stocks_df["growth_252d"] - 0.05)
    / stocks_df["volatility"]
)

In [78]:
print(stocks_df.columns)

Index(['Date', 'Ticker', 'Close', 'growth_252d', 'volatility', 'Sharpe'], dtype='object')


In [79]:
final = stocks_df[
    stocks_df["Date"] == "2026-09-11"
].copy()

In [80]:
final["Sharpe"] = final["Sharpe"].replace(
    [np.inf, -np.inf],
    np.nan
)

In [81]:
print(final["Sharpe"].describe())
print("Median Sharpe:", final["Sharpe"].median())

count    127.000000
mean       0.184895
std        0.578558
min       -0.040147
25%        0.010665
50%        0.046432
75%        0.119233
max        5.385326
Name: Sharpe, dtype: float64
Median Sharpe: 0.04643187288646578


In [82]:
stocks_df = stocks_df.sort_values(["Ticker", "Date"]).copy()

In [83]:
for month in range(1, 13):
    days = month * 21
    
    stocks_df[f"future_growth_{month}_m"] = (
        stocks_df.groupby("Ticker")["Close"]
        .transform(lambda x: x.shift(-days) / x)
    )

In [84]:
first_days = (
    stocks_df.groupby("Ticker")["Date"]
    .min()
    .reset_index(name="min_date")
)

print(first_days.head())

  Ticker   min_date
0   AAPG 2025-01-24
1   AARD 2025-02-13
2   ADVB 2025-03-06
3    AII 2025-05-08
4   AIRO 2025-06-13


In [85]:
entry_data = stocks_df.merge(
    first_days,
    left_on=["Ticker", "Date"],
    right_on=["Ticker", "min_date"],
    how="inner"
)

In [86]:
print(entry_data.shape)
print(entry_data.head())

(132, 19)
        Date Ticker      Close  growth_252d  volatility  Sharpe  \
0 2025-01-24   AAPG  17.379999          NaN         NaN     NaN   
1 2025-02-13   AARD  14.310000          NaN         NaN     NaN   
2 2025-03-06   ADVB  73.000000          NaN         NaN     NaN   
3 2025-05-08    AII  16.900000          NaN         NaN     NaN   
4 2025-06-13   AIRO  24.000000          NaN         NaN     NaN   

   future_growth_1_m  future_growth_2_m  future_growth_3_m  future_growth_4_m  \
0           1.159954           1.056962           1.512946           1.485328   
1           0.647100           0.555556           0.635220           0.758211   
2           0.920548           0.523288           0.255069           0.181370   
3           1.005917           1.020118           1.039645           1.156213   
4           1.085417           0.955833           0.841667           0.836667   

   future_growth_5_m  future_growth_6_m  future_growth_7_m  future_growth_8_m  \
0           2.34810

In [87]:
growth_columns = [
    f"future_growth_{month}_m"
    for month in range(1, 13)
]

stats = entry_data[growth_columns].describe()

print(stats)

       future_growth_1_m  future_growth_2_m  future_growth_3_m  \
count         132.000000         131.000000         131.000000   
mean           95.751507          67.978625          51.247805   
std          1087.893365         764.388723         573.269480   
min             0.072941           0.106197           0.090147   
25%             0.710433           0.584863           0.442477   
50%             0.935351           0.892958           0.827160   
75%             1.114199           1.164906           1.116547   
max         12500.000279        8750.000196        6562.500147   

       future_growth_4_m  future_growth_5_m  future_growth_6_m  \
count         131.000000         131.000000         131.000000   
mean           22.697288          54.242665          64.010844   
std           247.825831         609.321946         720.716387   
min             0.063927           0.032245           0.022075   
25%             0.392759           0.413434           0.308026   
50%      

In [88]:
median_growth = stats.loc["50%"]

print(median_growth)

future_growth_1_m     0.935351
future_growth_2_m     0.892958
future_growth_3_m     0.827160
future_growth_4_m     0.730400
future_growth_5_m     0.690909
future_growth_6_m     0.700916
future_growth_7_m     0.659973
future_growth_8_m     0.603508
future_growth_9_m     0.580294
future_growth_10_m    0.533333
future_growth_11_m    0.481811
future_growth_12_m    0.491838
Name: 50%, dtype: float64


In [89]:
best_month = median_growth.idxmax()
best_value = median_growth.max()

print("Best holding period:", best_month)
print("Maximum median growth:", best_value)

Best holding period: future_growth_1_m
Maximum median growth: 0.9353509333873746


In [90]:
!pip install gdown

In [92]:
import gdown
import pandas as pd

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "data.parquet",
    quiet=False
)

df = pd.read_parquet(
    "data.parquet",
    engine="pyarrow"
)

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=ef8ada77-4262-4758-85b5-4c52f1e59ead
To: C:\Users\EB-PC\data.parquet
100%|██████████| 130M/130M [02:15<00:00, 963kB/s]  


In [93]:
print(df.columns.tolist())

['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Year', 'Month', 'Weekday', 'Date', 'growth_1d', 'growth_3d', 'growth_7d', 'growth_30d', 'growth_90d', 'growth_365d', 'growth_future_30d', 'SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'is_positive_growth_30d_future', 'ticker_type', 'index_x', 'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'index_y', 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice', 'index', 'cdl2crows', '

In [94]:
selected_df = df[
    (df["rsi"] < 30) &
    (df["Date"] >= "2000-01-01") &
    (df["Date"] <= "2025-06-01")
].copy()

print(selected_df.shape)

(5206, 203)


In [95]:
net_income = 1000 * (
    selected_df["growth_future_30d"] - 1
).sum()

print("Net income: $", net_income)
print("Answer in $ thousands:", net_income / 1000)

Net income: $ 65805.58960747933
Answer in $ thousands: 65.80558960747933


In [ ]:
df.head()
df.columns

In [ ]:
close == running_max

In [ ]:
print(sp500.shape)
print(sp500.columns)
print(sp500['Close'].head())

In [ ]:
import yfinance as yf
import pandas as pd

sp500 = yf.download(
    '^GSPC',
    start='1950-01-01',
    end='2026-09-09',
    auto_adjust=False
)

print(sp500.shape)
print(sp500.head())
print(sp500.tail())

In [ ]:
sp500.head()

In [ ]:
type(sp500['Close'])

In [ ]:
close = sp500['Close'].squeeze()

In [ ]:
close.head()

In [ ]:
running_max = close.cummax()

In [ ]:
close == running_max

In [ ]:
ath = close[close == running_max]

In [ ]:
ath.head(20)

In [ ]:
ath_dates = ath.index

In [ ]:
corrections = []

for i in range(len(ath_dates) - 1):

    start_date = ath_dates[i]
    next_ath_date = ath_dates[i + 1]

    start_price = close.loc[start_date]

    period = close.loc[start_date:next_ath_date]

    low_price = period.min()
    low_date = period.idxmin()

    drawdown = (start_price - low_price) / start_price * 100

    duration = (low_date - start_date).days

    corrections.append({
        'Start Date': start_date,
        'ATH Price': start_price,
        'Low Date': low_date,
        'Low Price': low_price,
        'Next ATH Date': next_ath_date,
        'Drawdown %': drawdown,
        'Duration Days': duration
    })

In [ ]:
corrections_df = pd.DataFrame(corrections)

In [ ]:
corrections_df.head()

In [ ]:
significant = corrections_df[
    corrections_df['Drawdown %'] >= 5
]

In [ ]:
significant.head()

In [ ]:
largest = significant.sort_values(
    'Drawdown %',
    ascending=False
)

In [ ]:
largest.head(10)

In [ ]:
significant['Drawdown %'].median()

In [ ]:
significant['Drawdown %'].quantile([0.25, 0.50, 0.75])

In [ ]:
significant['Duration Days'].quantile(
    [0.25, 0.50, 0.75]
)

In [ ]:
import yfinance as yf
import pandas as pd

In [ ]:
ticker = 'AMZN'

ticker_obj = yf.Ticker(ticker)

earnings = ticker_obj.get_earnings_dates()

In [ ]:
earnings.head()

In [ ]:
earnings.shape

In [ ]:
earnings[['EPS Estimate', 'Reported EPS', 'Surprise(%)']]

In [ ]:
earnings = earnings.dropna(subset=['Surprise(%)'])

In [ ]:
earnings.shape

In [ ]:
print(earnings[['Reported EPS', 'Surprise(%)']])

In [ ]:
positive_surprises = earnings[
    earnings['Surprise(%)'] > 0
].copy()

In [ ]:
print(positive_surprises[['Reported EPS', 'Surprise(%)']])

In [ ]:
import yfinance as yf

amzn = yf.download(
    'AMZN',
    start='2020-01-01',
    end='2026-09-09',
    auto_adjust=False
)

In [ ]:
close = amzn['Close']['AMZN']

In [ ]:
print(close.head())

In [ ]:
two_day_return = close.shift(-2) / close - 1

In [ ]:
returns_df = pd.DataFrame({
    'Close': close,
    '2-Day Return': two_day_return
})

In [ ]:
print(returns_df.head())

In [ ]:
earnings.index = pd.to_datetime(earnings.index).tz_localize(None)

In [ ]:
returns_df.index = pd.to_datetime(returns_df.index).tz_localize(None)

In [ ]:
returns_df.index = pd.to_datetime(returns_df.index)

In [ ]:
combined = earnings[['Surprise(%)']].join(
    returns_df[['2-Day Return']],
    how='inner'
)

In [ ]:
print(combined)

In [ ]:
positive = combined[
    combined['Surprise(%)'] > 0
].copy()

In [ ]:
median_return = positive['2-Day Return'].median() * 100

print("Median 2-day return:", median_return, "%")

In [ ]:
correlation = positive[
    ['Surprise(%)', '2-Day Return']
].corr()

print(correlation)